# The Simplex Method, Step by Step
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Session 6 · about 35 min.*

> **How to use this notebook.** Run the cells from top to bottom (Shift+Enter).
> Cells marked **✍️ TODO** are for you; a **✅ check** right after tells you if your answer is correct.
> Cells marked **🤖 AI prompt** contain a prompt you can paste into Gemini (the ✨ button in Colab),
> ChatGPT, Claude… **Always read and test the code the AI gives you.**

The graphical method works with 2 variables. With 3 or more, we need an **algebraic** method: the
**simplex** walks from **corner to corner** of the feasible region, always to a better neighbour, one **pivot**
at a time, until no neighbour is better.

**The tableau.** One row per constraint (standard form: one slack per $\le$ row), labelled with its **basic
variable**; the last row is the objective written as $z - c\,x = 0$, so it holds $-c$. Basic variables take the
value of the RHS, non-basic variables are 0: that is the current corner.

**One iteration** (maximization):
1. **Optimal?** If no coefficient of the $z$ row is negative, stop.
2. **Entering variable — you choose:** a non-basic column with a **negative** coefficient in the $z$ row
   (any of them increases $z$; the usual rule takes the most negative one, leftmost on a tie).
3. **Leaving variable — the ratio test:** increase the entering variable; each row with a **positive**
   coefficient says how far it can go before its basic variable hits 0 (RHS ÷ coefficient). The smallest ratio
   leaves. No positive coefficient: the LP is **unbounded**.
4. **Pivot:** divide the pivot row by the pivot element, then make the rest of the column 0.

**Starting point.** With only $\le$ rows and RHS $\ge 0$, the origin is a corner: the slacks form the first basis.
With $\ge$ or $=$ rows it is not, and a **Phase I** finds a first corner (Part 3): the explorer detects it
automatically.

| Part | Topic | Time |
|------|-------|------|
| 1 | Demo: the carpenter | 5 min |
| 2 | ✍️ Activity: the coffee roaster (then the FA6 quiz) | 12 min |
| 3 | Demo: two phases, the Dorian problem | 6 min |
| 4 | ✍️ Activity: a two-phase problem | 8 min |

In [ ]:
# @title Setup and the simplex functions (run this cell; no need to read it)
import importlib, subprocess, sys
import importlib.metadata
try:                                   # PuLP 4 + its CBC solver (pulp[cbc]); replaces an older PuLP 3
    _old = int(importlib.metadata.version("pulp").split(".")[0]) < 4 or importlib.util.find_spec("cbcbox") is None
except importlib.metadata.PackageNotFoundError:
    _old = True
if _old:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[cbc]>=4"], check=True)
import base64, json, math, re
from fractions import Fraction as Fr
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pulp
from IPython.display import display, Markdown

_ANS = json.loads(base64.b64decode("eyJQX2VudGVyMSI6ICJlc3ByZXNzbyIsICJQX2xlYXZlMSI6ICJzX3JvYXN0IiwgIlBfejEiOiAzNTAwMCwgIlBfZW50ZXIyIjogImhvdXNlIiwgIlBfbGVhdmUyIjogInNfcGFjayIsICJQX3oyIjogMzYwMDAsICJQX2hvdXNlIjogMzAsICJQX2VzcHJlc3NvIjogMzAsICJQX2RlY2FmIjogMCwgIlBfYmVhbnNfbGVmdCI6IDEwLCAiUF9waXZvdHMiOiAyLCAiUV9waGFzZTEiOiAyLCAiUV96IjogIjE3LzIiLCAiUV94MSI6IDQsICJRX3gyIjogIjEvMiIsICJRX3gzIjogMCwgIlFfczIiOiAiNS8yIn0="))

def check(key, value):
    """Compare your answer with the expected one (variable names: exact; numbers: exact, "17/2" is fine)."""
    if value is None:
        print(f"⏳ {key}: not answered yet"); return
    exp = _ANS[key]
    if isinstance(exp, str) and not re.fullmatch(r"-?\d+(/\d+)?", exp):
        ok = str(value).strip().lower() == exp
    else:
        try:
            ok = Fr(str(value)).limit_denominator(10**6) == Fr(str(exp))
        except (ValueError, ZeroDivisionError):
            ok = False
    print(f"✅ {key}: correct" if ok else f"❌ {key}: not yet")

def solve_with_pulp(model):
    """The same model solved by a real solver (CBC), to compare with your tableau."""
    result = model.solve(pulp.COIN_CMD(msg=False))
    if result.status_str != "Optimal":
        print(f"PuLP: {result.status_str}"); return
    vals = ", ".join(f"{v.name} = {Fr(v.value()).limit_denominator(1000)}" for v in model.variables())
    print(f"PuLP: Optimal | {vals} | z = {Fr(pulp.value(model.objective)).limit_denominator(1000)}")

MAX_VARS, MAX_ROWS = 5, 5
SENSE = {pulp.LpConstraintLE: "<=", pulp.LpConstraintGE: ">=", pulp.LpConstraintEQ: "="}
FLIP = {"<=": ">=", ">=": "<=", "=": "="}

def read_model(model):
    """max/min c·x s.t. rows (a, sense, b) with b >= 0 and x >= 0, in exact fractions, read from a PuLP model."""
    xs, cons = model.variables(), model.constraints()
    if not xs or not cons or not model.objective or all(model.objective.get(v, 0) == 0 for v in xs):
        raise ValueError("The model has no objective or no constraints yet: did you fill in the ✍️ TODO lines?")
    if len(xs) > MAX_VARS or len(cons) > MAX_ROWS:
        raise ValueError(f"This explorer is for small models: at most {MAX_VARS} variables and {MAX_ROWS} constraints "
                         f"(yours has {len(xs)} and {len(cons)}).")
    for v in xs:
        if v.lowBound != 0 or (v.upBound is not None and not math.isinf(v.upBound)):
            raise ValueError(f"Declare {v.name} with lowBound=0 and no upBound (write an upper bound as a constraint).")
    q = lambda v: Fr(v).limit_denominator(10**6)
    c = [q(model.objective.get(v, 0)) for v in xs]
    rows, tags = [], []
    for i, con in enumerate(cons):
        a, s, b = [q(con.get(v, 0)) for v in xs], SENSE[con.sense], -q(con.constant)
        if b < 0:                                     # the simplex needs RHS >= 0: multiply the row by -1
            a, s, b = [-v for v in a], FLIP[s], -b
        rows.append((a, s, b))
        named = con.name and not con.name.startswith("_C")
        tags.append(f"_{con.name}" if named else str(i + 1))   # constraint "roast" -> slack s_roast; unnamed -> s1
    return dict(names=[v.name for v in xs], c=c, rows=rows, tags=tags, maximize=model.sense == pulp.LpMaximize)


_OUT = []                                             # output of the current run, printed at the end

def _emit(obj):
    _OUT.append(obj)

def _say(*args):
    _OUT.append(" ".join(str(a) for a in args))

def _show(T, cols, basis, label, col=None, row=None):
    """Tableau as a plain table of exact fractions (rows = basic variables, last row = objective)."""
    _emit(pd.DataFrame([[str(v) for v in r] for r in T], index=basis + [label], columns=cols + ["RHS"]))

def _times(a, name):
    return name if a == 1 else f"{a}·{name}"


def _pivot(T, r, k):
    T = [row[:] for row in T]
    p = T[r][k]
    T[r] = [v / p for v in T[r]]
    for i in range(len(T)):
        if i != r and T[i][k] != 0:
            f = T[i][k]
            T[i] = [a - f * b for a, b in zip(T[i], T[r])]
    return T


def simplex(model, enter=(), auto=False, plot=True):
    """Tableau simplex on a small PuLP model, in exact fractions; Phase I is added automatically when the origin
    is not a feasible starting point (>= or = rows). The whole procedure is printed, from the first tableau.
    enter: your entering variables, in order (one per pivot, Phase I and Phase II together).
    auto=True: the computer picks the most negative coefficient (leftmost on a tie) when your list is used up."""
    _OUT.clear()
    try:
        return _simplex(model, enter, auto, plot)
    finally:
        for obj in _OUT:
            (print if isinstance(obj, str) else display)(obj)


def _simplex(model, enter, auto, plot):
    try:
        lp = read_model(model)
    except ValueError as err:                          # an empty or too large model: a message, not a crash
        _emit(Markdown(f"⚠️ **{err}**"))
        return None
    n, m = len(lp["names"]), len(lp["rows"])
    COLS = list(lp["names"])
    T, basis, art = [[] for _ in range(m)], [], []
    for i, (a, s, b) in enumerate(lp["rows"]):        # standard form: slack +s, surplus -s, artificial +a
        T[i] = list(a)
    for i, (a, s, b) in enumerate(lp["rows"]):
        if s != "=":
            COLS.append("s" + lp["tags"][i])
            for k in range(m):
                T[k].append(Fr(int(k == i)) * (1 if s == "<=" else -1))
    for i, (a, s, b) in enumerate(lp["rows"]):
        if s != "<=":
            COLS.append("a" + lp["tags"][i]); art.append(len(COLS) - 1)
            for k in range(m):
                T[k].append(Fr(int(k == i)))
    for i, (a, s, b) in enumerate(lp["rows"]):
        T[i].append(b)
        basis.append(COLS.index(("s" if s == "<=" else "a") + lp["tags"][i]))

    enter, used, path, pivots = list(enter), 0, [], {"I": 0, "II": 0}
    two_phases = bool(art)
    sign = 1 if lp["maximize"] else -1                # we always MAXIMIZE: min z = max (-z)
    zlab = "z" if lp["maximize"] else "−z"

    def point():
        x = [Fr(0)] * len(COLS)
        for i, j in enumerate(basis):
            x[j] = T[i][-1]
        return x

    def report(phase, label):
        x = point()
        vals = ", ".join(f"{COLS[j]} = {x[j]}" for j in range(len(COLS)))
        obj = T[-1][-1]
        if phase == "I":
            tail = f"w = {-obj}" + ("  (all artificials are 0: a feasible corner)" if obj == 0 else
                                     "  (an artificial is > 0: not a solution of the original problem yet)")
        else:
            tail = f"z = {sign * obj}"
        _say(f"basic solution: {vals}  |  {tail}")
        path.append(([float(v) for v in x[:n]], phase))

    def iterate(phase, label):
        nonlocal T, used
        while True:
            cand = [j for j in range(len(COLS)) if j not in basis and j not in art and T[-1][j] < 0]
            it = pivots[phase]
            if not cand:
                _show(T, COLS, [COLS[j] for j in basis], label); report(phase, label)
                return "optimal"
            if used < len(enter):
                name = enter[used]
                if name not in COLS:
                    _show(T, COLS, [COLS[j] for j in basis], label); report(phase, label)
                    _emit(Markdown(f"⚠️ **`{name}` is not a column of this tableau.** Use one of {[COLS[j] for j in cand]}."))
                    return "stopped"
                k = COLS.index(name)
                if k not in cand:
                    _show(T, COLS, [COLS[j] for j in basis], label); report(phase, label)
                    why = "is basic" if k in basis else ("is artificial: once out, it never comes back" if k in art
                                                        else f"has coefficient {T[-1][k]} ≥ 0 in the {label} row "
                                                             f"({'w' if phase == 'I' else 'z'} would not improve)")
                    _emit(Markdown(f"⚠️ **{name} {why}.** Choose one of {[COLS[j] for j in cand]} "
                                     f"(non-basic, negative coefficient in the {label} row)."))
                    return "stopped"
                used += 1
            elif auto:
                k = min(cand, key=lambda j: (T[-1][j], j))
            else:
                _show(T, COLS, [COLS[j] for j in basis], label); report(phase, label)
                _emit(Markdown(f"**Your turn:** add one of {[COLS[j] for j in cand]} to your list "
                                 f"(non-basic, negative coefficient in the {label} row: each one improves "
                                 f"{'w' if phase == 'I' else 'z'}), then run the cell again."))
                return "waiting"
            ratios = {i: T[i][-1] / T[i][k] for i in range(m) if T[i][k] > 0}
            if not ratios:
                _show(T, COLS, [COLS[j] for j in basis], label, col=k); report(phase, label)
                _emit(Markdown(f"**{COLS[k]} enters**, but its column has **no positive coefficient**: {COLS[k]} can "
                                 "grow forever and every constraint stays satisfied. **The LP is unbounded.**"))
                return "unbounded"
            r = min(ratios, key=lambda i: (ratios[i], i))
            _show(T, COLS, [COLS[j] for j in basis], label, col=k, row=r); report(phase, label)
            e = COLS[k]
            why = []                                   # the ratio test, row by row
            for i in range(m):
                a, b, bv = T[i][k], T[i][-1], COLS[basis[i]]
                if a > 0:
                    why.append(f"* {bv} = {b} − {_times(a, e)} ≥ 0  ⇒  {e} ≤ {b} ÷ {a} = **{ratios[i]}**")
                else:
                    why.append(f"* {bv} = {b} + {_times(-a, e) if a else '0·' + e}: coefficient {a} ≤ 0, "
                               f"{bv} never reaches 0, no limit on {e}")
            tie = "; tie, so the topmost row" if list(ratios.values()).count(ratios[r]) > 1 else ""
            _emit(Markdown(f"**{e} enters** (your choice: coefficient {T[-1][k]} in the {label} row). "
                           f"Increase {e} (the other non-basic variables stay 0) while every basic variable stays ≥ 0:\n"
                           + "\n".join(why)
                           + f"\n\n**{COLS[basis[r]]} reaches 0 first (smallest ratio {ratios[r]}{tie}): it leaves**, "
                           f"{e} takes its place in the basis. Pivot = {T[r][k]}: divide its row by {T[r][k]}, then "
                           f"make the rest of the {e} column 0."))
            T = _pivot(T, r, k)
            basis[r] = k
            pivots[phase] += 1
            _emit(Markdown(f"#### Iteration {pivots['I'] + pivots['II']}"
                           + (f" (Phase {phase})" if two_phases else "")))

    status = None
    if art:
        _emit(Markdown("### Phase I: find a first corner\n"
                         f"Artificial variables {[COLS[j] for j in art]} give a starting basis. We maximize "
                         "**−w = −(sum of the artificials)**, i.e. we minimize w. The −w row starts with +1 under each "
                         "artificial; subtracting the artificial rows makes it 0 under the basic artificials:"))
        T.append([Fr(int(j in art)) for j in range(len(COLS))] + [Fr(0)])
        _show(T, COLS, [COLS[j] for j in basis], "−w")
        for i, j in enumerate(basis):
            if j in art:
                T[-1] = [a - b for a, b in zip(T[-1], T[i])]
        _emit(Markdown("#### Iteration 0 (Phase I): first tableau (−w row priced out)"))
        status = iterate("I", "−w")
        if status != "optimal":
            return dict(status=status, pivots=pivots)
        if T[-1][-1] < 0:
            _emit(Markdown(f"❌ **Phase I ends with w\\* = {-T[-1][-1]} > 0**: the artificials cannot all reach 0, "
                             "so no point satisfies every constraint. **The LP is infeasible.**"))
            if plot and n == 2:
                plot_path(lp, path)
            return dict(status="infeasible", pivots=pivots, w=-T[-1][-1])
        for i in range(m - 1, -1, -1):                 # an artificial still basic at 0: pivot it out or drop the row
            if basis[i] in art:
                k = next((j for j in range(len(COLS)) if j not in art and j not in basis and T[i][j] != 0), None)
                if k is None:
                    _emit(Markdown(f"Row of {COLS[basis[i]]} is redundant (a combination of the others): dropped."))
                    del T[i]; del basis[i]; m -= 1
                else:
                    _emit(Markdown(f"{COLS[basis[i]]} is still basic at 0: a pivot on {COLS[k]} replaces it (z unchanged)."))
                    T = _pivot(T, i, k); basis[i] = k
        keep = [j for j in range(len(COLS)) if j not in art]
        T = [[row[j] for j in keep] + [row[-1]] for row in T[:-1]]
        basis = [keep.index(j) for j in basis]
        COLS = [COLS[j] for j in keep]
        art = []
        _emit(Markdown("✅ **Phase I done: w\\* = 0**, the basis is a feasible corner. Drop the artificial columns.\n\n"
                         "### Phase II: optimize z from this corner\n"
                         f"Write the {zlab} row again ({'−c' if lp['maximize'] else '+c'} under the decision "
                         "variables). It must be 0 under every basic variable: subtract (coefficient × row) "
                         "for each basic variable whose coefficient is not 0."))
        T.append([-sign * v for v in lp["c"]] + [Fr(0)] * (len(COLS) - n) + [Fr(0)])
        _show(T, COLS, [COLS[j] for j in basis], zlab)
        for i, j in enumerate(basis):
            f = T[-1][j]
            if f != 0:
                T[-1] = [a - f * b for a, b in zip(T[-1], T[i])]
        _emit(Markdown(f"#### Iteration {pivots['I']} (Phase II): first tableau ({zlab} row priced out)"))
    else:
        if not lp["maximize"]:
            _emit(Markdown("**Minimization:** min z = max (−z). The −z row holds +c; the most negative "
                             "coefficient still enters, and z = −(RHS of the −z row)."))
        T.append([-sign * v for v in lp["c"]] + [Fr(0)] * (len(COLS) - n) + [Fr(0)])
        _emit(Markdown("All constraints are ≤ with RHS ≥ 0: the slacks form the first basis (the origin), "
                       "no Phase I needed.\n\n#### Iteration 0: first tableau"))
    status = iterate("II", zlab)
    x = path[-1][0] if path else None
    if status == "optimal":
        _emit(Markdown(f"✅ **No negative coefficient in the {zlab} row: optimal.** z = {sign * T[-1][-1]} "
                         + (f"({pivots['I']} pivot(s) in Phase I, {pivots['II']} in Phase II)." if two_phases
                            else f"after {pivots['II']} pivot(s).")))
    if plot and n == 2:
        plot_path(lp, path)
    vals = dict(zip(COLS, [Fr(0)] * len(COLS)))
    vals.update({COLS[j]: T[i][-1] for i, j in enumerate(basis)})
    return dict(status=status, z=sign * T[-1][-1], values=vals, pivots=pivots)


def plot_path(lp, path):
    """Feasible region of a 2-variable model and the corners visited (Phase I dashed, Phase II solid)."""
    A = np.array([[float(v) for v in a] for a, _, _ in lp["rows"]]); b = np.array([float(r[2]) for r in lp["rows"]])
    S = [s for _, s, _ in lp["rows"]]
    pts = [p for p, _ in path]
    hi = max([max(p) for p in pts] + [b[i] / A[i, j] for i in range(len(b)) for j in range(2) if A[i, j] > 0] + [1])
    hi *= 1.15
    X, Y = np.meshgrid(np.linspace(0, hi, 600), np.linspace(0, hi, 600))
    ok = np.ones_like(X, bool)
    for i, s in enumerate(S):
        lhs = A[i, 0] * X + A[i, 1] * Y
        ok &= (lhs <= b[i] + 1e-9) if s == "<=" else (lhs >= b[i] - 1e-9) if s == ">=" else np.isclose(lhs, b[i], atol=hi / 300)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    if ok.any():
        ax.contourf(X, Y, ok, levels=[0.5, 1], colors=["#0072CE"], alpha=0.18)
    xs = np.linspace(0, hi, 200)
    for i in range(len(b)):
        terms = [f"{'' if a == 1 else '−' if a == -1 else f'{a:g}'}{v}" for a, v in zip(A[i], lp["names"]) if a != 0]
        lab = " + ".join(terms).replace("+ −", "− ").replace("+ -", "− ") + f" {S[i].replace('<=', '≤').replace('>=', '≥')} {b[i]:g}"
        if A[i, 1] != 0:
            ax.plot(xs, (b[i] - A[i, 0] * xs) / A[i, 1], lw=1, label=lab)
        else:
            ax.axvline(b[i] / A[i, 0], lw=1, label=lab)
    for ph, style, col, lab in [("I", "--o", "#E07B00", "Phase I"), ("II", "-o", "#007A4C", "simplex path")]:
        seg = [p for p, f in path if f == ph]
        if ph == "II" and any(f == "I" for _, f in path) and seg:
            seg = [[p for p, f in path if f == "I"][-1]] + seg
        if seg:
            P = np.array(seg); ax.plot(P[:, 0], P[:, 1], style, color=col, lw=2, ms=6, zorder=4, label=lab)
    ax.set_xlim(-0.03 * hi, hi); ax.set_ylim(-0.03 * hi, hi)
    ax.set_xlabel(lp["names"][0]); ax.set_ylabel(lp["names"][1]); ax.legend(fontsize=7, loc="upper right")
    plt.close(fig)
    _emit(fig)
print("Setup done ✔")

---
## 1 · Demo: the carpenter
$\max\ z = 500t + 300c$ s.t. time $2t + c \le 40$, material $t + 2c \le 50$, $t, c \ge 0$. In PuLP:

In [ ]:
E = pulp.LpProblem("carpenter", pulp.LpMaximize)
t = E.add_variable("t", lowBound=0)
c = E.add_variable("c", lowBound=0)
E += 500 * t + 300 * c
E += 2 * t + c <= 40
E += t + 2 * c <= 50
print(E)

**You decide, the computer computes.** `simplex(model, enter)` starts from the **first tableau** and takes your
**ordered list of entering variables**: the first name enters at iteration 1, the second at iteration 2, …
For each one, the computer does the ratio test (and explains it), picks the leaving variable, pivots, and prints
the new tableau: **every run prints the whole procedure**. When your list is used up, it tells you which
variables could enter next, or that the tableau is optimal ✅. For 2 variables it also draws the path.

Build the list one name at a time, re-running the same cell:
* `E_enter = []`: the first tableau. The $z$ row is $(-500, -300)$: both $t$ and $c$ can enter. Take $t$ (most negative).
* `E_enter = ["t"]`: $t$ increases until $s_1$ hits 0 ($t \le 20$ before $t \le 50$): $s_1$ leaves, corner $(20, 0)$,
  $z = 10{,}000$. Only $c$ is still negative ($-50$).
* `E_enter = ["t", "c"]`: $s_2$ leaves; no negative coefficient left: optimal ✅.

In [ ]:
E_enter = ["t", "c"]
resE = simplex(E, E_enter)

**Things to notice.**
* Path $(0, 0) \to (20, 0) \to (10, 20)$, $z = 11{,}000$: the same optimum as the graphical method, after
  visiting only three corners.
* Try `E_enter = ["c", "t"]`: another path, $(0, 0) \to (0, 25) \to (10, 20)$, **same optimum**.
  The choice changes the route, not the destination.
* Try `E_enter = ["s1"]`: refused, $s_1$ is basic.

---
## 2 · ✍️ Activity: the coffee roaster
A roaster produces batches of three blends (3 variables: no graph possible). Profit and resources per batch:

| | House | Espresso | Decaf | Available per week |
|---|---|---|---|---|
| Profit (\$/batch) | 500 | 700 | 500 | |
| Roasting (h) | 2 | 3 | 2 | 150 |
| Packaging (h) | 1 | 1 | 2 | 60 |
| Green beans (units of 10 kg) | 1 | 1 | 1 | 70 |

The model is ready (next cell). Its constraints are named, so the slacks are `s_roast`, `s_pack`, `s_beans`.
1. Build `P_enter` **one name at a time**, always taking the **most negative** coefficient of the $z$ row
   (leftmost on a tie): start with `[]`, run, read the $z$ row, add a name, run again… until ✅ optimal.
2. Read the procedure to answer the checks below, then the **FA6 quiz** in Canvas.

In [ ]:
P = pulp.LpProblem("roaster", pulp.LpMaximize)
house = P.add_variable("house", lowBound=0)
espresso = P.add_variable("espresso", lowBound=0)
decaf = P.add_variable("decaf", lowBound=0)
P += 500 * house + 700 * espresso + 500 * decaf
P += 2 * house + 3 * espresso + 2 * decaf <= 150, "roast"
P += house + espresso + 2 * decaf <= 60, "pack"
P += house + espresso + decaf <= 70, "beans"

In [ ]:
P_enter = []             # ✍️ TODO: add the entering variables one at a time, e.g. ["house"], then ["house", ...]
resP = simplex(P, P_enter)

### ✍️ Read the procedure (write names as strings, e.g. `"house"`, `"s_pack"`)

In [ ]:
P_enter1 = None          # ✍️ entering variable at iteration 1
P_leave1 = None          # ✍️ leaving variable at iteration 1
P_z1 = None              # ✍️ z after iteration 1
P_enter2 = None          # ✍️ entering variable at iteration 2 (a tie: the leftmost one)
P_leave2 = None          # ✍️ leaving variable at iteration 2
P_z2 = None              # ✍️ z after iteration 2
P_house = None           # ✍️ batches of House in the optimal plan
P_espresso = None        # ✍️ batches of Espresso
P_decaf = None           # ✍️ batches of Decaf (a non-basic variable is 0)
P_beans_left = None      # ✍️ unused green beans at the optimum (value of the slack s_beans)
P_pivots = None          # ✍️ number of pivots
for k in ["P_enter1", "P_leave1", "P_z1", "P_enter2", "P_leave2", "P_z2",
          "P_house", "P_espresso", "P_decaf", "P_beans_left", "P_pivots"]:
    check(k, globals()[k])

Check with a solver: PuLP must find the same plan.

In [ ]:
solve_with_pulp(P)

---
## 3 · Demo: two phases (the Dorian problem)
$\min\ z = 50x_1 + 100x_2$ s.t. $7x_1 + 2x_2 \ge 28$, $2x_1 + 12x_2 \ge 24$, $x_1, x_2 \ge 0$.

**Why the origin does not work.** A $\ge$ row gets a surplus with $-1$: $7x_1 + 2x_2 - s_1 = 28$ gives
$s_1 = -28 < 0$ at $x = 0$. **Fix:** add an artificial $a_i \ge 0$ to each $\ge$ or $=$ row
($7x_1 + 2x_2 - s_1 + a_1 = 28$); the artificials give a first basis, but a point with some $a_i > 0$ is not a
solution of the real problem.
* **Phase I:** minimize $w = $ sum of the artificials (we maximize $-w$: same rules). The $-w$ row starts with
  $+1$ under each artificial and is made 0 under the basic artificials. $w^* = 0$: a **feasible corner**.
* **Phase II:** drop the artificial columns, write the real objective row again, make it 0 under the basic
  variables, and continue as usual. (If $w^* > 0$: the LP is **infeasible**.)
* **Minimization:** min $z$ = max $(-z)$: the row is labelled $-z$, the same rules apply, $z = -$RHS.

The explorer sees the $\ge$ rows and adds Phase I **automatically**; your list covers both phases. Built step by step:
* `[]`: the Phase I tableau, $-w$ row $(-9, -14, 1, 1)$: $x_2$ is the most negative.
* `["x2"]`: $a_2$ leaves; $w = 24$ still $> 0$; now $x_1$ ($-\tfrac{20}{3}$).
* `["x2", "x1"]`: $a_1$ leaves, $w = 0$: Phase I ends, Phase II starts and is already optimal ✅.

In [ ]:
D = pulp.LpProblem("Dorian", pulp.LpMinimize)
x1 = D.add_variable("x1", lowBound=0)
x2 = D.add_variable("x2", lowBound=0)
D += 50 * x1 + 100 * x2
D += 7 * x1 + 2 * x2 >= 28
D += 2 * x1 + 12 * x2 >= 24
D_enter = ["x2", "x1"]
resD = simplex(D, D_enter)

**Things to notice.**
* Phase I starts at $(0, 0)$, **outside** the feasible region (dashed orange path), and walks to the corner
  $(\tfrac{18}{5}, \tfrac75)$ where $w = 0$.
* There, the restored $-z$ row has no negative coefficient: Phase II needs **no pivot**, $z = 320$. Usually
  Phase II has work to do, as in Part 4.

---
## 4 · ✍️ Activity: a two-phase problem
$$\max\ z = 2x_1 + x_2 + x_3 \quad \text{s.t.}\quad 3x_1 + 4x_2 + 2x_3 \le 14,\;\; 2x_1 + x_2 + x_3 \ge 6,\;\;
x_1 + x_3 = 4,\;\; x_1, x_2, x_3 \ge 0$$
1. **Before running:** which rows get a slack, a surplus, an artificial? What is the first basis?
2. The model is ready (next cell). Build `Q_enter` one name at a time, always the **most negative**
   coefficient (leftmost on a tie), through Phase I and Phase II, until ✅ optimal.

In [ ]:
Q = pulp.LpProblem("practice", pulp.LpMaximize)
x1 = Q.add_variable("x1", lowBound=0)
x2 = Q.add_variable("x2", lowBound=0)
x3 = Q.add_variable("x3", lowBound=0)
Q += 2 * x1 + x2 + x3
Q += 3 * x1 + 4 * x2 + 2 * x3 <= 14
Q += 2 * x1 + x2 + x3 >= 6
Q += x1 + x3 == 4

In [ ]:
Q_enter = []             # ✍️ TODO: add the entering variables one at a time (Phase I first, then Phase II)
resQ = simplex(Q, Q_enter)

In [ ]:
Q_phase1 = None          # ✍️ number of pivots in Phase I
Q_z = None               # ✍️ optimal z (a fraction is fine: "17/2")
Q_x1 = None              # ✍️ optimal x1
Q_x2 = None              # ✍️ optimal x2
Q_x3 = None              # ✍️ optimal x3
Q_s2 = None              # ✍️ surplus of constraint 2 at the optimum
for k in ["Q_phase1", "Q_z", "Q_x1", "Q_x2", "Q_x3", "Q_s2"]:
    check(k, globals()[k])

## 🚀 If you have time
* **Unbounded:** $\max\ x_1 + x_2$ s.t. $x_1 - x_2 \le 1$. Run `simplex(U, ["x1", "x2"])` on the model below:
  at iteration 2 the column of $x_2$ has no positive coefficient, so $x_2$ can grow forever.
* **Infeasible:** add `U += x1 + x2 >= 5` and `U += x1 + x2 <= 3`, then run `simplex(U, auto=True)`:
  Phase I ends with $w^* > 0$.
* Re-run the roaster with `P_enter = ["house", ...]` (another valid choice): how many pivots now?

In [ ]:
U = pulp.LpProblem("unbounded", pulp.LpMaximize)
x1 = U.add_variable("x1", lowBound=0)
x2 = U.add_variable("x2", lowBound=0)
U += x1 + x2
U += x1 - x2 <= 1
simplex(U, ["x1", "x2"]);

---
### Take-aways
* Each pivot moves to an **adjacent corner** with a better (or equal) $z$; the simplex stops when the objective
  row has no negative coefficient.
* **You** choose who enters (any negative coefficient works: the path changes, the optimum does not); the
  **ratio test** decides who leaves, so the solution stays feasible.
* Basic variables = RHS, non-basic = 0. A basic slack > 0 means a resource is not fully used.
* $\ge$ or $=$ rows: **Phase I** finds a first corner ($w^* = 0$) or proves the LP infeasible ($w^* > 0$).
* No positive coefficient in the entering column: the LP is **unbounded**.